Model Setup

In [ ]:
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

import dynestyx as dsx
from dynestyx import DynamicalModel


def hmm_model(A=None, obs_times=None, obs_values=None, predict_times=None):
    A = numpyro.sample("A", dist.Dirichlet(jnp.ones(2)).expand([2]).to_event(1), obs=A)

    def state_evolution(x, u, t_now, t_next):
        return dist.Categorical(probs=A[x])

    def observation_model(x, u, t):
        probs = jnp.array(
            [
                [1 / 6, 1 / 6, 1 / 6, 1 / 6, 1 / 6, 1 / 6],
                [1 / 10, 1 / 10, 1 / 10, 1 / 10, 1 / 10, 1 / 2],
            ]
        )
        return dist.Categorical(probs=probs[x])

    dynamics = DynamicalModel(
        initial_condition=dist.Categorical(probs=jnp.ones(2) / 2),
        state_evolution=state_evolution,
        observation_model=observation_model,
    )

    return dsx.sample(
        "f",
        dynamics,
        obs_times=obs_times,
        obs_values=obs_values,
        predict_times=predict_times,
    )

Generating data (same logic)

In [ ]:
import jax.random as jr
from numpyro.infer import Predictive

from dynestyx import DiscreteTimeSimulator, flatten_draws

n_rollout_eval = 100
n_train = 10000
obs_times_full = jnp.arange(start=0.0, stop=n_train + n_rollout_eval, step=1.0)
obs_times = obs_times_full[:n_train]
rollout_eval_times = obs_times_full[n_train:]

prng_key = jr.PRNGKey(0)
predictive_model = Predictive(hmm_model, num_samples=1)

true_A = jnp.array([[0.95, 0.05], [0.1, 0.9]])

with DiscreteTimeSimulator():
    synthetic_samples = predictive_model(prng_key, A=true_A, predict_times=obs_times_full)

# Extract observations from synthetic data.
# f_states / f_observations have shape (num_samples, n_sim, T, 1) — the trailing
# singleton is the state_dim convention used for all simulators.  Index [0, 0, :, 0]
# to recover a 1-D array suitable for plotting and conditioning.
print(
    "synthetic shapes:",
    synthetic_samples["f_times"].shape,
    synthetic_samples["f_states"].shape,
    synthetic_samples["f_observations"].shape,
)
obs_all = synthetic_samples["f_observations"][0, 0, :, 0]
states_all = synthetic_samples["f_states"][0, 0, :, 0]
obs_values = obs_all[:n_train]
obs_values_eval_rollout = obs_all[n_train:]

states_true = states_all[:n_train]
states_true_eval_rollout = states_all[n_train:]


There is an utility dynestyx provides that allows us to plot HMM observations

In [ ]:
from dynestyx.diagnostics.plotting_utils import plot_hmm_states_and_observations

plot_hmm_states_and_observations(
    times=obs_times[:100],
    x=states_true[:100],
    y=obs_values[:100],
)

Code for Bayesian inference on HMM

In [ ]:
from numpyro.infer import MCMC, NUTS
from dynestyx import Filter
from dynestyx.inference.filters import HMMConfig

mcmc_key = jr.PRNGKey(0)

with Filter(filter_config=HMMConfig()):
    nuts_kernel = NUTS(hmm_model)
    mcmc = MCMC(
        nuts_kernel,
        num_samples=500,
        num_warmup=500,
    )
    mcmc.run(mcmc_key, obs_times=obs_times, obs_values=obs_values)

posterior_samples = mcmc.get_samples()

import matplotlib.pyplot as plt
import seaborn as sns

plt.hist(
    posterior_samples["A"][:, 0, 1],
    bins=20,
    color="k",
    alpha=0.5,
    label=r"p0→1",
)
plt.axvline(true_A[0, 1], color="k", linestyle="--")
plt.hist(
    posterior_samples["A"][:, 1, 0],
    bins=20,
    color="b",
    alpha=0.5,
    label=r"p1→0",
)
plt.axvline(true_A[1, 0], color="b", linestyle="--")
sns.despine()
plt.legend()
plt.show()

We can evaluate filter + simulator similarly